# 11. GNINA 도킹 — 다양성 선별 1,000개

3단계에서 고른 1,000개를 CDK2(3RPR) 포켓에 도킹한다.

설정은 02·03 단계에서 **실측으로 확정한 것**을 그대로 쓴다.
- 박스: center (92.37, 80.29, −47.95), size 16.1 × 15.8 × 16.5
- exhaustiveness 8, num_modes 9, seed 1, CNN rescore 켬
- 근거: CNN 채점 비용이 사실상 0(59.0초 vs 60.6초), 박스 16/20/24 Å 차이 0.13 kcal/mol,
  redocking top-pose RMSD 0.60 Å(결정 좌표) / 0.56 Å(SMILES → 새 3D)

실행 환경: Colab 무료 (T4). 개당 약 59초 → 1,000개는 약 16시간이므로 **나흘에 나눠 돌린다.**

## 끊김 대비
- 리간드 한 개가 끝날 때마다 결과를 Drive 의 CSV 에 **즉시 추가**한다
- 다시 실행하면 이미 끝난 dock_id 는 건너뛴다 (셀 상태에 의존하지 않고 CSV 를 읽는다)
- `MINUTES_BUDGET` 만큼만 돌고 스스로 멈춘다. 런타임이 끊기기 전에 정리하기 위한 것


## 1. Drive 연결


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, re, json, time, glob, site, subprocess, shutil
from pathlib import Path

ROOT  = Path('/content/drive/MyDrive/cdk2')
TOOLS = ROOT / 'tools'
EXP   = ROOT / 'experiment'
SPEED = EXP / '02_gnina_speed'
OUT   = EXP / '11_docking'
POSES = OUT / 'poses'
POSES.mkdir(parents=True, exist_ok=True)

GNINA    = TOOLS / 'gnina'
RECEPTOR = SPEED / 'receptor.pdb'

# 02·03 단계에서 확정한 도킹 박스 — 바꾸지 않는다
BOX = dict(center_x=92.37, center_y=80.29, center_z=-47.95,
           size_x=16.1,    size_y=15.8,    size_z=16.5)

EXHAUSTIVENESS, NUM_MODES, GNINA_SEED = 8, 9, 1

# 이 세션에서 몇 분까지 돌릴지 (하루 5시간 = 300분)
# 996개 예상 19~22시간 — 도킹 시간이 분자 크기에 비례하고(회전결합 1개당 +9.9초)
# 선별 분자가 속도 측정셋보다 크다(회전결합 4.83 vs 4.11). 4시간×4일로는 모자란다.
MINUTES_BUDGET = 300

for p in [GNINA, RECEPTOR]:
    print(('OK   ' if p.exists() else 'MISSING ') + str(p))
print('출력 폴더:', OUT)


## 2. GNINA 실행 환경

gnina 릴리스 바이너리는 CUDA 빌드라 CUDA 라이브러리가 필요하다.
Colab 이미지가 Python 3.13 으로 바뀌면서 nvidia 라이브러리가 기본 포함되지 않으므로,
`ldd` 로 **모자란 것만** 골라 pip 설치하고 `LD_LIBRARY_PATH` 에 추가한다.
(03 단계 노트북과 같은 셀이다.)


In [ ]:
os.chmod(GNINA, 0o755)

gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or '없음 → 런타임 유형을 T4 GPU로 바꾸세요')

PKG = {
    'libcudart':     'nvidia-cuda-runtime-cu12',
    'libcufft':      'nvidia-cufft-cu12',
    'libcublas':     'nvidia-cublas-cu12',
    'libcublasLt':   'nvidia-cublas-cu12',
    'libcurand':     'nvidia-curand-cu12',
    'libcusparse':   'nvidia-cusparse-cu12',
    'libcusolver':   'nvidia-cusolver-cu12',
    'libcudnn':      'nvidia-cudnn-cu12',
    'libnvrtc':      'nvidia-cuda-nvrtc-cu12',
    'libnvJitLink':  'nvidia-nvjitlink-cu12',
    'libnvToolsExt': 'nvidia-nvtx-cu12',
    'libcupti':      'nvidia-cuda-cupti-cu12',
    'libnccl':       'nvidia-nccl-cu12',
}

def lib_dirs():
    dirs = []
    for sp in site.getsitepackages():
        dirs += glob.glob(f'{sp}/nvidia/*/lib')
    dirs += glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib')
    dirs += ['/usr/local/cuda/lib64', '/usr/lib64-nvidia']
    return sorted({d for d in dirs if os.path.isdir(d)})

def make_env():
    env = dict(os.environ)
    env['LD_LIBRARY_PATH'] = ':'.join(lib_dirs() + [os.environ.get('LD_LIBRARY_PATH', '')]).strip(':')
    return env

def missing_libs(env):
    r = subprocess.run(['ldd', str(GNINA)], capture_output=True, text=True, env=env)
    return re.findall(r'^\s*(\S+?)\.so[\.\d]*\s*=>\s*not found', r.stdout, re.M)

GNINA_ENV = make_env()
installed = set()
for attempt in range(8):
    miss = missing_libs(GNINA_ENV)
    if not miss:
        break
    todo = {PKG[m] for m in miss if m in PKG} - installed
    unknown = [m for m in miss if m not in PKG]
    if unknown:
        print('매핑에 없는 라이브러리:', unknown)
    if not todo:
        print('설치할 수 있는 패키지가 없습니다. 부족한 것:', miss)
        break
    print(f'[{attempt+1}] 부족: {miss} → 설치: {sorted(todo)}')
    subprocess.run(['pip', '-q', 'install', *sorted(todo)])
    installed |= todo
    GNINA_ENV = make_env()

os.environ['LD_LIBRARY_PATH'] = GNINA_ENV['LD_LIBRARY_PATH']

r = subprocess.run([str(GNINA), '--version'], capture_output=True, text=True, env=GNINA_ENV)
print()
print(r.stdout.strip() or r.stderr.strip())
print('\n✅ gnina 준비 완료' if r.returncode == 0 else '\n⚠ gnina 실행 실패')


## 3. 리간드 3D 구조

**저장소에 올려 둔 SDF 를 그대로 받아 쓴다.**
리간드 구조는 `scripts/10_ligand_prep.py` 로 로컬에서 한 번 생성해
`experiment/10_ligands/ligands_all.sdf.gz` 로 커밋해 두었다.

여기서 다시 만들지 않는 이유: ETKDG 는 시드를 고정해도 **RDKit 버전이 다르면**
좌표가 달라질 수 있다. 도킹 입력을 파일로 고정해 두면 어디서 돌리든 같은 입력이 된다.
(내려받지 못할 때만 같은 스크립트로 즉석 생성하는 경로를 남겨 둔다.)

생성 조건: `ETKDGv3` randomSeed=42, MMFF94s maxIters=2000 수렴 확인,
수소를 뗀 SDF (03 단계 redocking B 와 같은 형식).


In [ ]:
!pip -q install rdkit 2>/dev/null | tail -1
import gzip
import pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

RAW = 'https://raw.githubusercontent.com/jang0920-eng/2026_deeplearning/main'
SEL_URL = f'{RAW}/experiment/09_diversity/09_selected_for_docking.csv'
SDF_URL = f'{RAW}/experiment/10_ligands/ligands_all.sdf.gz'
PREP_URL = f'{RAW}/scripts/10_ligand_prep.py'

sel = pd.read_csv(SEL_URL)
print(f'선별 분자 {len(sel):,}개 (RF 점수 {sel.score.min():.3f} ~ {sel.score.max():.3f})')

LIG_SDF = OUT / 'ligands_all.sdf'

if not LIG_SDF.exists():
    gz = Path('/content/ligands_all.sdf.gz')
    r = subprocess.run(['curl', '-sSL', '-f', '-o', str(gz), SDF_URL])
    if r.returncode == 0 and gz.stat().st_size > 1000:
        with gzip.open(gz, 'rt') as fi:
            LIG_SDF.write_text(fi.read())
        print('저장소의 리간드 SDF 를 받았습니다:', LIG_SDF)
    else:
        print('SDF 를 받지 못해 같은 스크립트로 직접 생성합니다 (약 20분)')
        subprocess.run(['curl', '-sSL', '-o', '/content/lig_prep.py', PREP_URL], check=True)
        import importlib.util
        spec = importlib.util.spec_from_file_location('lig_prep', '/content/lig_prep.py')
        lig_prep = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(lig_prep)

        w, rows = Chem.SDWriter(str(LIG_SDF)), []
        t0 = time.time()
        for i, r in enumerate(sel.itertuples(index=False), 1):
            m, status, energy = lig_prep.build_3d(r.smiles)
            rows.append(dict(dock_id=r.dock_id, chembl_id=r.chembl_id, smiles=r.smiles,
                             ml_score=r.score, status=status, mmff_energy=energy))
            if m is not None:
                m.SetProp('_Name', r.dock_id)
                for k, v in [('dock_id', r.dock_id), ('chembl_id', r.chembl_id),
                             ('ml_score', f'{r.score:.4f}'), ('smiles', r.smiles)]:
                    m.SetProp(k, str(v))
                w.write(m)
            if i % 100 == 0:
                print(f'  {i:,}/{len(sel):,}  {time.time()-t0:.0f}초', flush=True)
        w.close()
        pd.DataFrame(rows).to_csv(OUT / '11_ligand_prep_log.csv', index=False)
        print('생성 완료:', LIG_SDF)
else:
    print('Drive 에 있는 리간드 SDF 를 사용합니다:', LIG_SDF)

LIGANDS = [m for m in Chem.SDMolSupplier(str(LIG_SDF)) if m is not None]
print(f'도킹 대상 {len(LIGANDS):,}개')


## 4. 도킹 루프

- 리간드 하나가 끝나면 결과를 Drive CSV 에 **바로 추가**한다
- 이미 CSV 에 있는 dock_id 는 건너뛴다 → 세션이 끊겨도 이어서 돌릴 수 있다
- 세션 첫 도킹은 CNN 모델 로딩 때문에 비정상적으로 느리다(~180초). **속도 집계에서 제외**한다


In [ ]:
RESULTS = OUT / '11_docking_results.csv'
FIELDS = ['dock_id', 'chembl_id', 'ml_score', 'vina_top', 'cnn_score_top',
          'cnn_affinity_top', 'vina_best', 'n_poses', 'seconds', 'session_first', 'smiles']

def load_done():
    if not RESULTS.exists():
        return set()
    return set(pd.read_csv(RESULTS)['dock_id'].astype(str))

def append_row(row):
    new = not RESULTS.exists()
    with open(RESULTS, 'a') as f:
        if new:
            f.write(','.join(FIELDS) + '\n')
        f.write(','.join('' if row.get(k) is None else str(row.get(k)) for k in FIELDS) + '\n')

def dock_one(mol, out_sdf):
    lig = Path('/content/lig.sdf')
    w = Chem.SDWriter(str(lig)); w.write(mol); w.close()
    cmd = [str(GNINA), '-r', str(RECEPTOR), '-l', str(lig), '-o', str(out_sdf),
           '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(NUM_MODES),
           '--seed', str(GNINA_SEED)]
    for k, v in BOX.items():
        cmd += [f'--{k}', str(v)]
    res = subprocess.run(cmd, capture_output=True, text=True, env=GNINA_ENV)
    if res.returncode != 0:
        return None, res.stderr[-500:]
    return out_sdf, None

def read_poses(sdf):
    def prop(m, name):
        return float(m.GetProp(name)) if m.HasProp(name) else None
    poses = [m for m in Chem.SDMolSupplier(str(sdf), removeHs=False) if m is not None]
    if not poses:
        return None
    vina = [prop(m, 'minimizedAffinity') for m in poses]
    return dict(vina_top=vina[0], cnn_score_top=prop(poses[0], 'CNNscore'),
                cnn_affinity_top=prop(poses[0], 'CNNaffinity'),
                vina_best=min([v for v in vina if v is not None], default=None),
                n_poses=len(poses))

done = load_done()
todo = [m for m in LIGANDS if m.GetProp('dock_id') not in done]
print(f'완료 {len(done):,}개 / 남음 {len(todo):,}개')

t_start, times, n_ok, n_fail = time.time(), [], 0, 0
for i, mol in enumerate(todo, 1):
    if (time.time() - t_start) / 60 > MINUTES_BUDGET:
        print(f'\n⏹ 이 세션 예산 {MINUTES_BUDGET}분을 다 썼습니다. 다음 세션에서 이어서 실행하세요.')
        break

    did = mol.GetProp('dock_id')
    out_sdf = POSES / f'{did}.sdf'
    t0 = time.time()
    sdf, err = dock_one(mol, out_sdf)
    dt = round(time.time() - t0, 1)

    row = dict(dock_id=did, chembl_id=mol.GetProp('chembl_id'),
               ml_score=mol.GetProp('ml_score'), seconds=dt,
               session_first=int(i == 1), smiles=mol.GetProp('smiles'))
    if sdf is None:
        n_fail += 1
        print(f'  [{i}/{len(todo)}] {did} 실패: {err}')
    else:
        p = read_poses(sdf)
        if p:
            row.update(p)
        n_ok += 1
        if i > 1:
            times.append(dt)
    append_row(row)

    if i % 10 == 0 or i == 1:
        med = sorted(times)[len(times)//2] if times else float('nan')
        rest = len(todo) - i
        print(f'  [{i}/{len(todo)}] {did}  {dt}초  중앙값 {med:.1f}초  '
              f'남은 예상 {med*rest/3600:.1f}시간', flush=True)

print(f'\n이 세션: 성공 {n_ok}  실패 {n_fail}  경과 {(time.time()-t_start)/60:.1f}분')
if times:
    s = sorted(times)
    print(f'개당 중앙값 {s[len(s)//2]:.1f}초 (첫 도킹 제외, n={len(s)})')


## 5. 중간 집계

지금까지 끝난 것만으로 집계한다. 1,000개가 다 끝나지 않아도 돌려 볼 수 있다.


In [ ]:
res = pd.read_csv(RESULTS).drop_duplicates('dock_id', keep='last')
ok = res.dropna(subset=['vina_top'])
print(f'완료 {len(res):,}개 / 점수 확보 {len(ok):,}개 / 실패 {len(res)-len(ok):,}개')

t = res[res.session_first == 0]['seconds'].dropna()
if len(t):
    print(f'개당 중앙값 {t.median():.1f}초 (세션 첫 도킹 제외)')

print('\nVina 점수 분포 (낮을수록 좋음)')
print(ok['vina_top'].describe().round(3).to_string())
print(f"\n양수 점수(결합 불성립): {(ok['vina_top'] > 0).sum()}개")

print('\nCNNaffinity 분포 (높을수록 좋음)')
print(ok['cnn_affinity_top'].describe().round(3).to_string())

top20 = ok.sort_values('cnn_affinity_top', ascending=False).head(20)
print('\nCNNaffinity 상위 20')
print(top20[['dock_id', 'chembl_id', 'ml_score', 'vina_top', 'cnn_affinity_top']]
      .to_string(index=False))


## 6. 다음 단계로 넘길 목록

도킹 점수는 **순위를 정하는 도구**이고 실험으로 확인된 결합 친화도가 아니다.
여기서는 5단계(ADMET)로 넘길 약 100개를 추린다.

두 점수를 함께 쓴다.
- **Vina 점수** — 물리 기반 점수 함수
- **CNNaffinity** — GNINA 의 3D CNN 점수 (W3 Conv3d). 서로 독립적인 평가다

두 순위를 모두 만족하는 분자를 고르면 한쪽 점수 함수의 치우침에 덜 의존한다.


In [ ]:
N_NEXT = 100

ok = ok.copy()
ok['rank_vina'] = ok['vina_top'].rank(ascending=True)          # 낮을수록 좋음
ok['rank_cnn']  = ok['cnn_affinity_top'].rank(ascending=False) # 높을수록 좋음
ok['rank_mean'] = (ok['rank_vina'] + ok['rank_cnn']) / 2

sel_next = ok.sort_values('rank_mean').head(N_NEXT)
sel_next.to_csv(OUT / '11_top_for_admet.csv', index=False)
print(f'저장: {OUT / "11_top_for_admet.csv"}  ({len(sel_next)}개)')
print(sel_next[['dock_id', 'chembl_id', 'vina_top', 'cnn_affinity_top', 'rank_mean']]
      .head(20).to_string(index=False))

# 두 점수가 얼마나 같은 방향을 보는지 — 보고서에 쓸 숫자
print('\nVina vs CNNaffinity 순위 상관 (Spearman):',
      round(ok[['vina_top', 'cnn_affinity_top']].corr(method='spearman').iloc[0, 1], 3))


## 7. skill 문서용 요약 생성

Solar Pro4 가 CSV 를 직접 읽지 못할 수 있으므로 숫자를 **텍스트 표**로 뽑아 둔다.


In [ ]:
lines = []
lines.append('### 4단계 GNINA 도킹 (다양성 선별 1,000개)\n')
lines.append(f'- 박스: center ({BOX["center_x"]}, {BOX["center_y"]}, {BOX["center_z"]}), '
             f'size ({BOX["size_x"]}, {BOX["size_y"]}, {BOX["size_z"]})')
lines.append(f'- 설정: exhaustiveness {EXHAUSTIVENESS}, num_modes {NUM_MODES}, '
             f'seed {GNINA_SEED}, CNN rescore 켬')
lines.append(f'- 리간드 3D: ETKDGv3 randomSeed=42, MMFF94s maxIters=2000 수렴 확인\n')
lines.append(f'| 항목 | 값 |')
lines.append('|---|---|')
lines.append(f'| 도킹 완료 | {len(res):,}개 |')
lines.append(f'| 점수 확보 | {len(ok):,}개 |')
lines.append(f'| 실패 | {len(res)-len(ok):,}개 |')
if len(t):
    lines.append(f'| 개당 중앙값 | {t.median():.1f}초 (세션 첫 도킹 제외) |')
    lines.append(f'| 총 계산 시간 | {res["seconds"].sum()/3600:.1f}시간 |')
lines.append(f'| Vina 점수 중앙값 | {ok["vina_top"].median():.2f} kcal/mol |')
lines.append(f'| Vina 최고 | {ok["vina_top"].min():.2f} kcal/mol |')
lines.append(f'| CNNaffinity 중앙값 | {ok["cnn_affinity_top"].median():.2f} |')
lines.append(f'| CNNaffinity 최고 | {ok["cnn_affinity_top"].max():.2f} |')
lines.append(f'| 양수 Vina 점수 | {(ok["vina_top"] > 0).sum()}개 |')
lines.append(f'\n두 점수의 순위 상관(Spearman): '
             f'{ok[["vina_top","cnn_affinity_top"]].corr(method="spearman").iloc[0,1]:.3f}')
lines.append('\n도킹 점수는 순위를 정하는 도구이며 실험으로 검증된 결합 친화도가 아니다.')

txt = '\n'.join(lines)
(OUT / '11_docking_summary.md').write_text(txt, encoding='utf-8')
print(txt)
